# Entrainement from scratch du modele de resume automatique

Avant de lancer ce notebook: `Execution > Modifier le type d'execution > GPU`.

In [ ]:
import torch

print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os

if os.path.exists('/content/memoir/.git'):
    %cd /content/memoir
    !git pull
else:
    %cd /content
    !git clone https://github.com/hous2020/memoir.git
    %cd /content/memoir

In [ ]:
!pip install -r requirements.txt

## Datasets disponibles

| Dataset | Exemples (train) | Description |
|---|---|---|
| `xlsum` | ~38k | Articles BBC en français |
| `mlsum` | ~33k | Articles de presse français |
| `wiki_lingua_fr` | ~41k | Articles Wikipedia FR avec résumé |
| **`all`** | **~112k** | **Tous combinés (recommandé)** |

> Tous chargés via fichiers Parquet directs — pas de script de chargement, compatible avec toutes les versions de `datasets`.

In [ ]:
# Test rapide : tokenizer sur 1000 exemples par dataset (5000 total)
!python src/train_tokenizer.py --dataset-name all --max-samples 5000 --vocab-size 30000

In [ ]:
# Test rapide : modèle sur 1000 exemples par dataset, 3 epochs
!python src/train_scratch.py --dataset-name all --max-samples 5000 --epochs 3 --batch-size 32

In [ ]:
!python src/evaluation.py --model-type scratch --dataset-name xlsum --num-samples 20

## Entraînement complet — tous les datasets

Deux niveaux selon le GPU disponible.

**GPU T4 (gratuit)** : ~20k exemples/dataset = 100k total, 10 epochs (~3-4h)  
**GPU A100 (Colab Pro)** : ~100k exemples/dataset = 500k total, 15 epochs (~8-12h)

In [ ]:
# ── GPU T4 (Colab gratuit) ──────────────────────────────────────────────────
# ~20k exemples par dataset = 100k total | architecture moyenne | ~3-4h
!python src/train_tokenizer.py \
    --dataset-name all \
    --max-samples 100000 \
    --vocab-size 32000

!python src/train_scratch.py \
    --dataset-name all \
    --max-samples 100000 \
    --epochs 10 \
    --batch-size 32 \
    --d-model 384 \
    --nhead 6 \
    --num-layers 4 \
    --learning-rate 0.0001 \
    --max-seq-len 128

# ── GPU A100 (Colab Pro) — décommenter pour utiliser ────────────────────────
# ~100k exemples par dataset = 500k total | architecture optimale | ~8-12h
# !python src/train_tokenizer.py \
#     --dataset-name all \
#     --max-samples 500000 \
#     --vocab-size 40000

# !python src/train_scratch.py \
#     --dataset-name all \
#     --max-samples 500000 \
#     --epochs 15 \
#     --batch-size 64 \
#     --d-model 512 \
#     --nhead 8 \
#     --num-layers 6 \
#     --learning-rate 0.00005 \
#     --max-seq-len 150

## Sauvegarde dans Google Drive

Lance ces cellules si tu veux conserver le modele apres la session Colab.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/memoir_models
# !cp models/transformer_scratch.pth /content/drive/MyDrive/memoir_models/
# !cp data/custom_tokenizer.json /content/drive/MyDrive/memoir_models/